<a href="https://colab.research.google.com/github/SANGHATI23/ohdsi-fhir-omop-showcase-demo/blob/main/FHIRy_pyOMOP_TFL_SubmissionHardening_v12_P0_1_P0_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FHIRy–pyOMOP TFL

1. Freeze one complete V0 source archive and deterministic V1–V5 source packages.
2. Freeze a versioned Transformation Fidelity Layer schema and warning vocabulary.



# Phase A

## Environment and paths

In [1]:
from pathlib import Path
from collections import defaultdict
import hashlib
import json
import shutil
import tarfile
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")

candidates = [
    Path("/content/drive/MyDrive/MyDrive fhir_omop_colab "),
    Path("/content/drive/MyDrive/MyDrive fhir_omop_colab"),
]

RAW_ROOT = next((p for p in candidates if p.exists()), None)

if RAW_ROOT is None:
    matches = [
        p for p in MYDRIVE.iterdir()
        if p.is_dir() and "fhir_omop_colab" in p.name
    ]
    if len(matches) == 1:
        RAW_ROOT = matches[0]
    else:
        raise FileNotFoundError("Could not resolve controlled-source root.")

RUN_ROOT = MYDRIVE / "fhir_omop_colab" / "tfl_execution_v6"
FREEZE_ROOT = RUN_ROOT / "submission_freeze_v12"
SOURCE_FREEZE_DIR = FREEZE_ROOT / "sources"
SCHEMA_DIR = FREEZE_ROOT / "schema"
MANIFEST_DIR = FREEZE_ROOT / "manifests"

for p in [FREEZE_ROOT, SOURCE_FREEZE_DIR, SCHEMA_DIR, MANIFEST_DIR]:
    p.mkdir(parents=True, exist_ok=True)

ARCHIVES = {
    "V1": RAW_ROOT / "V1_missing_demographics_clinical_core_25k.tar.gz",
    "V2": RAW_ROOT / "V2_duplicate_encounter_ids_clinical_core_25k.tar.gz",
    "V3": RAW_ROOT / "V3_conflicting_codings_clinical_core_25k.tar.gz",
    "V4": RAW_ROOT / "V4_missing_medications_clinical_core_25k.tar.gz",
}

for variant, path in ARCHIVES.items():
    if not path.exists():
        raise FileNotFoundError(path)

print("RAW_ROOT:", RAW_ROOT)
print("FREEZE_ROOT:", FREEZE_ROOT)

Mounted at /content/drive
RAW_ROOT: /content/drive/MyDrive/MyDrive fhir_omop_colab 
FREEZE_ROOT: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/submission_freeze_v12


# Phase B

## Read controlled FHIR archives

In [2]:
def canonical_json(resource):
    return json.dumps(
        resource,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    )

def read_fhir_archive(archive_path):
    grouped = defaultdict(list)

    with tarfile.open(archive_path, "r:gz") as tar:
        members = [
            m for m in tar.getmembers()
            if m.isfile() and m.name.lower().endswith(".ndjson")
        ]

        if not members:
            raise RuntimeError(f"No NDJSON files found in {archive_path}")

        for member in members:
            handle = tar.extractfile(member)
            if handle is None:
                continue

            for raw_line in handle:
                line = raw_line.decode("utf-8").strip()
                if not line:
                    continue

                obj = json.loads(line)

                if obj.get("resourceType") == "Bundle":
                    for entry in obj.get("entry", []):
                        resource = entry.get("resource", {})
                        rt = resource.get("resourceType")
                        if rt:
                            grouped[str(rt)].append(resource)
                else:
                    rt = obj.get("resourceType")
                    if rt:
                        grouped[str(rt)].append(obj)

    return dict(grouped)

RAW_VARIANTS = {
    variant: read_fhir_archive(path)
    for variant, path in ARCHIVES.items()
}

inventory_rows = []
for variant, grouped in RAW_VARIANTS.items():
    for resource_type, rows in sorted(grouped.items()):
        inventory_rows.append({
            "variant": variant,
            "resource_type": resource_type,
            "rows": len(rows),
        })

inventory = pd.DataFrame(inventory_rows)
display(
    inventory.pivot_table(
        index="resource_type",
        columns="variant",
        values="rows",
        aggfunc="sum",
        fill_value=0,
    )
)

variant,V1,V2,V3,V4
resource_type,,,,
Condition,25000,25000,25000,25000
Encounter,25000,25000,25000,25000
Medication,14131,14131,14131,12718
MedicationAdministration,14131,14131,14131,12718
MedicationRequest,25000,25000,25000,25000
Observation,25000,25000,25000,25000
Patient,1071,1071,1071,1071
Procedure,25000,25000,25000,25000


## Verify unaffected-domain consensus

In [3]:
def variant_affects_resource_type(variant, resource_type):
    if variant == "V1":
        return resource_type == "Patient"
    if variant == "V2":
        return resource_type == "Encounter"
    if variant == "V3":
        return resource_type == "Condition"
    if variant == "V4":
        return resource_type.startswith("Medication")
    return False

ALL_RESOURCE_TYPES = sorted(
    set().union(
        *[set(grouped.keys()) for grouped in RAW_VARIANTS.values()]
    )
)

def ordered_domain_digest(resources):
    h = hashlib.sha256()
    for resource in resources:
        h.update(canonical_json(resource).encode("utf-8"))
        h.update(b"\n")
    return h.hexdigest()

V0_RESOURCES = {}
consensus_rows = []

for resource_type in ALL_RESOURCE_TYPES:
    donors = [
        variant
        for variant in ["V1", "V2", "V3", "V4"]
        if not variant_affects_resource_type(variant, resource_type)
        and resource_type in RAW_VARIANTS[variant]
    ]

    if len(donors) < 2:
        raise RuntimeError(
            f"{resource_type}: fewer than two unaffected donors."
        )

    summaries = pd.DataFrame([
        {
            "variant": variant,
            "rows": len(RAW_VARIANTS[variant][resource_type]),
            "digest": ordered_domain_digest(
                RAW_VARIANTS[variant][resource_type]
            ),
        }
        for variant in donors
    ])

    if summaries["rows"].nunique() != 1 or summaries["digest"].nunique() != 1:
        display(summaries)
        raise RuntimeError(
            f"{resource_type}: unaffected donor domains are not identical."
        )

    selected = donors[0]
    V0_RESOURCES[resource_type] = [
        json.loads(canonical_json(resource))
        for resource in RAW_VARIANTS[selected][resource_type]
    ]

    consensus_rows.append({
        "resource_type": resource_type,
        "selected_donor": selected,
        "verified_donors": "|".join(donors),
        "rows": len(V0_RESOURCES[resource_type]),
        "sha256": ordered_domain_digest(V0_RESOURCES[resource_type]),
    })

V0_CONSENSUS = pd.DataFrame(consensus_rows)
display(V0_CONSENSUS)

print("PASS: every V0 domain is supported by identical unaffected controls.")

,resource_type,selected_donor,verified_donors,rows,sha256
0,Condition,V1,V1|V2|V4,25000,e2080aa4ab64f70f311327d2f5889dedad4c895695600d...
1,Encounter,V1,V1|V3|V4,25000,e5a6865450aa409dbcdfa4d779505d4270320b6540fccb...
2,Medication,V1,V1|V2|V3,14131,eac792bcfc32e37037155b6f88e87adf788417f10a1e36...
3,MedicationAdministration,V1,V1|V2|V3,14131,4431fe39a3ed6a3f5f080dd5475a22bfa769d254a0cde5...
4,MedicationRequest,V1,V1|V2|V3,25000,2e93e03b21726b2fee472fecbdc500c7916b5ace307096...
5,Observation,V1,V1|V2|V3|V4,25000,d16c265866bc167a136399f02412b82918f40c1e1f58c1...
6,Patient,V2,V2|V3|V4,1071,d5434fd061e66a58b924cb05f01cac86b5b92a396af6ab...
7,Procedure,V1,V1|V2|V3|V4,25000,c7bc967ed07da418b738c704f784d8265ffae123cbc396...


PASS: every V0 domain is supported by identical unaffected controls.


# Phase C

## Freeze V0 and V1–V5 source packages

In [4]:
def write_resource_directory(grouped_resources, out_dir):
    out_dir = Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for resource_type, rows in sorted(grouped_resources.items()):
        path = out_dir / f"{resource_type}.ndjson"
        with open(path, "w", encoding="utf-8") as handle:
            for resource in rows:
                handle.write(json.dumps(resource, ensure_ascii=False) + "\n")

    return out_dir

def make_tar_gz(source_dir, archive_path):
    source_dir = Path(source_dir)
    archive_path = Path(archive_path)
    if archive_path.exists():
        archive_path.unlink()

    with tarfile.open(archive_path, "w:gz") as tar:
        for path in sorted(source_dir.glob("*.ndjson")):
            tar.add(path, arcname=path.name)

    return archive_path

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def deep_copy_grouped(grouped):
    return {
        rt: [json.loads(canonical_json(resource)) for resource in rows]
        for rt, rows in grouped.items()
    }

FROZEN_VARIANTS = {
    variant: deep_copy_grouped(V0_RESOURCES)
    for variant in ["V0", "V1", "V2", "V3", "V4", "V5"]
}

FROZEN_VARIANTS["V1"]["Patient"] = deep_copy_grouped(
    {"Patient": RAW_VARIANTS["V1"]["Patient"]}
)["Patient"]

FROZEN_VARIANTS["V2"]["Encounter"] = deep_copy_grouped(
    {"Encounter": RAW_VARIANTS["V2"]["Encounter"]}
)["Encounter"]

FROZEN_VARIANTS["V3"]["Condition"] = deep_copy_grouped(
    {"Condition": RAW_VARIANTS["V3"]["Condition"]}
)["Condition"]

for resource_type in ALL_RESOURCE_TYPES:
    if resource_type.startswith("Medication"):
        if resource_type in RAW_VARIANTS["V4"]:
            FROZEN_VARIANTS["V4"][resource_type] = deep_copy_grouped(
                {resource_type: RAW_VARIANTS["V4"][resource_type]}
            )[resource_type]
        else:
            FROZEN_VARIANTS["V4"].pop(resource_type, None)

FROZEN_VARIANTS["V5"]["Patient"] = deep_copy_grouped(
    {"Patient": RAW_VARIANTS["V1"]["Patient"]}
)["Patient"]

FROZEN_VARIANTS["V5"]["Condition"] = deep_copy_grouped(
    {"Condition": RAW_VARIANTS["V3"]["Condition"]}
)["Condition"]

EXPECTED_CHANGED_DOMAINS = {
    "V0": set(),
    "V1": {"Patient"},
    "V2": {"Encounter"},
    "V3": {"Condition"},
    "V4": {
        rt for rt in ALL_RESOURCE_TYPES
        if rt.startswith("Medication")
    },
    "V5": {"Patient", "Condition"},
}

def domain_signature(grouped, resource_type):
    rows = grouped.get(resource_type, [])
    return len(rows), ordered_domain_digest(rows)

validation_rows = []

for variant in ["V0", "V1", "V2", "V3", "V4", "V5"]:
    changed = {
        rt
        for rt in ALL_RESOURCE_TYPES
        if domain_signature(FROZEN_VARIANTS[variant], rt)
        != domain_signature(FROZEN_VARIANTS["V0"], rt)
    }

    if changed != EXPECTED_CHANGED_DOMAINS[variant]:
        raise RuntimeError(
            f"{variant}: changed domains {sorted(changed)} do not match "
            f"expected {sorted(EXPECTED_CHANGED_DOMAINS[variant])}."
        )

    validation_rows.append({
        "variant": variant,
        "changed_domains": "|".join(sorted(changed)),
        "validation_pass": True,
    })

archive_rows = []

for variant, grouped in FROZEN_VARIANTS.items():
    out_dir = SOURCE_FREEZE_DIR / f"{variant}_frozen"
    out_archive = (
        SOURCE_FREEZE_DIR
        / f"{variant}_frozen_clinical_core_25k.tar.gz"
    )

    write_resource_directory(grouped, out_dir)
    make_tar_gz(out_dir, out_archive)

    archive_rows.append({
        "variant": variant,
        "archive": out_archive.name,
        "total_resources": sum(len(rows) for rows in grouped.values()),
        "resource_types": len(grouped),
        "sha256": sha256_file(out_archive),
        "size_mb": out_archive.stat().st_size / (1024**2),
    })

FROZEN_ARCHIVE_MANIFEST = pd.DataFrame(archive_rows)
VARIANT_DOMAIN_VALIDATION = pd.DataFrame(validation_rows)

display(FROZEN_ARCHIVE_MANIFEST)
display(VARIANT_DOMAIN_VALIDATION)

FROZEN_ARCHIVE_MANIFEST.to_csv(
    MANIFEST_DIR / "frozen_source_archives_v12.csv",
    index=False,
)
V0_CONSENSUS.to_csv(
    MANIFEST_DIR / "v0_domain_consensus_v12.csv",
    index=False,
)
VARIANT_DOMAIN_VALIDATION.to_csv(
    MANIFEST_DIR / "variant_domain_validation_v12.csv",
    index=False,
)

print("PASS: V0-V5 frozen source packages and hashes written.")

,variant,archive,total_resources,resource_types,sha256,size_mb
0,V0,V0_frozen_clinical_core_25k.tar.gz,154333,8,31d5d00d1da8fef25cff0019c02806a997367a4a0ac2aa...,8.298301
1,V1,V1_frozen_clinical_core_25k.tar.gz,154333,8,2f0930aaa6d09727fdb6ef87a88b3e6a85dd1ab3251b53...,8.297845
2,V2,V2_frozen_clinical_core_25k.tar.gz,154333,8,71fedeb65cdadd06a2d3ecf836f9945b9b3655117ed7ed...,8.295615
3,V3,V3_frozen_clinical_core_25k.tar.gz,154333,8,f7127fb8aedc02015b4f1dcf03a49219966f616554edfd...,8.315295
4,V4,V4_frozen_clinical_core_25k.tar.gz,151507,8,1fed37d1ef19058fc6d11d2e6e7ec90ac83431db290c13...,8.216201
5,V5,V5_frozen_clinical_core_25k.tar.gz,154333,8,ecab4f72fac91ee204555a9d47d48d7dfb252ee1805989...,8.314818


,variant,changed_domains,validation_pass
0,V0,,True
1,V1,Patient,True
2,V2,Encounter,True
3,V3,Condition,True
4,V4,Medication|MedicationAdministration|Medication...,True
5,V5,Condition|Patient,True


PASS: V0-V5 frozen source packages and hashes written.


# Phase D

## Freeze the TFL schema and vocabularies

In [5]:
TFL_SCHEMA_VERSION = "1.0.0"

FIDELITY_STATUS_VALUES = [
    "PRESERVED",
    "NORMALIZED",
    "COLLAPSED",
    "AMBIGUOUS",
    "UNMAPPED",
    "DROPPED",
    "TRACEABILITY_LOSS",
]

WARNING_DEFINITIONS = [
    ("W_DEMOGRAPHIC_MISSING", "Prespecified demographic information is missing."),
    ("W_DUPLICATE_SOURCE_ID", "Configured source identifier is duplicated."),
    ("W_TRACEABILITY_LOSS", "Source identity cannot be uniquely recovered through lineage."),
    ("W_CONFLICTING_CODING", "Competing coding representation is detected by the configured deterministic rule."),
    ("W_UNMAPPED_RESOURCE", "Source resource has no supported target mapping."),
    ("W_MEDICATION_ATTRIBUTION", "Medication attribution cannot be resolved deterministically."),
    ("W_UNRESOLVED_REFERENCE", "Required FHIR reference cannot be resolved."),
]

WARNING_CODES = {row[0] for row in WARNING_DEFINITIONS}

TFL_REQUIRED_FIELDS = [
    "transformation_id",
    "source_resource_type",
    "source_resource_id",
    "source_reference_or_path",
    "target_omop_table",
    "target_omop_record_id",
    "target_omop_field",
    "mapping_rule_id",
    "fidelity_status",
    "warning_code",
    "transformation_version",
]

TFL_JSON_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$id": "https://example.org/tfl/schema/1.0.0",
    "title": "Transformation Fidelity Layer audit event",
    "type": "object",
    "required": TFL_REQUIRED_FIELDS,
    "properties": {
        "transformation_id": {"type": "string", "minLength": 1},
        "source_resource_type": {"type": "string", "minLength": 1},
        "source_resource_id": {"type": ["string", "null"]},
        "source_reference_or_path": {"type": ["string", "null"]},
        "source_value_or_code": {
            "type": ["string", "number", "boolean", "null"]
        },
        "target_omop_table": {"type": ["string", "null"]},
        "target_omop_record_id": {
            "type": ["string", "integer", "number", "null"]
        },
        "target_omop_field": {"type": ["string", "null"]},
        "mapping_rule_id": {"type": ["string", "null"]},
        "fidelity_status": {
            "type": "string",
            "enum": FIDELITY_STATUS_VALUES,
        },
        "warning_code": {"type": ["string", "null"]},
        "transformation_version": {"type": "string", "minLength": 1},
    },
    "additionalProperties": True,
}

schema_path = SCHEMA_DIR / "tfl_audit_schema_v1.0.0.json"
warning_path = SCHEMA_DIR / "tfl_warning_vocabulary_v1.0.0.csv"
status_path = SCHEMA_DIR / "tfl_fidelity_status_v1.0.0.csv"

schema_path.write_text(
    json.dumps(TFL_JSON_SCHEMA, indent=2),
    encoding="utf-8",
)

pd.DataFrame(
    WARNING_DEFINITIONS,
    columns=["warning_code", "description"],
).to_csv(warning_path, index=False)

pd.DataFrame({
    "fidelity_status": FIDELITY_STATUS_VALUES,
}).to_csv(status_path, index=False)

print("Schema:", schema_path)
print("Warning vocabulary:", warning_path)
print("Status vocabulary:", status_path)

Schema: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/submission_freeze_v12/schema/tfl_audit_schema_v1.0.0.json
Warning vocabulary: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/submission_freeze_v12/schema/tfl_warning_vocabulary_v1.0.0.csv
Status vocabulary: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/submission_freeze_v12/schema/tfl_fidelity_status_v1.0.0.csv


## Validate current corrected audits against the frozen schema

In [6]:
V10_AUDIT_DIR = (
    RUN_ROOT
    / "encounter_audit_repair_v10"
    / "fidelity_audit_corrected"
)

if not V10_AUDIT_DIR.exists():
    raise FileNotFoundError(V10_AUDIT_DIR)

def split_warning_codes(value):
    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    text = str(value).strip()

    if not text or text.lower() in {"nan", "none", "null"}:
        return []

    return [item for item in text.split("|") if item]

schema_validation_rows = []

for variant in ["V0", "V1", "V2", "V3", "V4", "V5"]:
    path = V10_AUDIT_DIR / f"{variant}_fidelity_audit_v10.parquet"

    if not path.exists():
        raise FileNotFoundError(path)

    audit = pd.read_parquet(path)

    missing_required = [
        field
        for field in TFL_REQUIRED_FIELDS
        if field not in audit.columns
    ]

    if missing_required:
        raise RuntimeError(
            f"{variant}: missing required TFL fields: {missing_required}"
        )

    invalid_status = sorted(
        set(audit["fidelity_status"].dropna().astype(str))
        - set(FIDELITY_STATUS_VALUES)
    )

    if invalid_status:
        raise RuntimeError(
            f"{variant}: invalid fidelity_status values: {invalid_status}"
        )

    observed_warning_codes = set()
    for value in audit["warning_code"]:
        observed_warning_codes.update(split_warning_codes(value))

    unknown_warnings = sorted(observed_warning_codes - WARNING_CODES)

    if unknown_warnings:
        raise RuntimeError(
            f"{variant}: unknown warning codes: {unknown_warnings}"
        )

    duplicate_transformation_ids = int(
        audit["transformation_id"]
        .astype(str)
        .duplicated(keep=False)
        .sum()
    )

    if duplicate_transformation_ids:
        raise RuntimeError(
            f"{variant}: duplicated transformation_id rows: "
            f"{duplicate_transformation_ids:,}"
        )

    schema_validation_rows.append({
        "variant": variant,
        "audit_rows": len(audit),
        "required_fields_present": True,
        "fidelity_status_valid": True,
        "warning_vocabulary_valid": True,
        "transformation_id_unique": True,
    })

TFL_SCHEMA_VALIDATION = pd.DataFrame(schema_validation_rows)
display(TFL_SCHEMA_VALIDATION)

TFL_SCHEMA_VALIDATION.to_csv(
    MANIFEST_DIR / "tfl_schema_validation_v12.csv",
    index=False,
)

print("PASS: corrected v10 audits satisfy the frozen TFL schema.")

,variant,audit_rows,required_fields_present,fidelity_status_valid,warning_vocabulary_valid,transformation_id_unique
0,V0,179333,True,True,True,True
1,V1,179333,True,True,True,True
2,V2,179333,True,True,True,True
3,V3,179333,True,True,True,True
4,V4,176507,True,True,True,True
5,V5,179333,True,True,True,True


PASS: corrected v10 audits satisfy the frozen TFL schema.


# Phase E

## Freeze manifest

In [7]:
manifest = {
    "freeze_version": "submission-freeze-v12",
    "tfl_schema_version": TFL_SCHEMA_VERSION,
    "source_root": str(RAW_ROOT),
    "frozen_source_manifest": str(
        MANIFEST_DIR / "frozen_source_archives_v12.csv"
    ),
    "v0_domain_consensus": str(
        MANIFEST_DIR / "v0_domain_consensus_v12.csv"
    ),
    "variant_domain_validation": str(
        MANIFEST_DIR / "variant_domain_validation_v12.csv"
    ),
    "tfl_schema": str(schema_path),
    "warning_vocabulary": str(warning_path),
    "fidelity_status_vocabulary": str(status_path),
}

manifest_path = MANIFEST_DIR / "submission_freeze_v12.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print(manifest_path)
print(manifest_path.read_text(encoding="utf-8"))

/content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/submission_freeze_v12/manifests/submission_freeze_v12.json
{
  "freeze_version": "submission-freeze-v12",
  "tfl_schema_version": "1.0.0",
  "source_root": "/content/drive/MyDrive/MyDrive fhir_omop_colab ",
  "frozen_source_manifest": "/content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/submission_freeze_v12/manifests/frozen_source_archives_v12.csv",
  "v0_domain_consensus": "/content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/submission_freeze_v12/manifests/v0_domain_consensus_v12.csv",
  "variant_domain_validation": "/content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/submission_freeze_v12/manifests/variant_domain_validation_v12.csv",
  "tfl_schema": "/content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/submission_freeze_v12/schema/tfl_audit_schema_v1.0.0.json",
  "warning_vocabulary": "/content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/submission_freeze_v12/schema/tfl_warning_vocabulary_v1.0.0.csv",
  "fidel

# Exit criteria

This notebook passes only when the clean V0 package is frozen and hashed, V1–V5 differ from V0 only in prespecified controlled domains, and the current corrected audits satisfy a versioned TFL schema and warning vocabulary.

